# Chapter 06 답안 양식. 데이터를 보며 질문을 만드는 EDA

> 이 내용을 `chapter06.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 박재선
- GitHub ID: lilypark0930-lab
- 작성일: 2026.10.18
- 최종 제출 URL: https://github.com/lilypark0930-lab/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb


## 1. 내가 정한 EDA 질문

| 질문 | 분석 범위 | 지표 | 검증 방법 |
| --- | --- | --- | --- |
| 카테고리별 completed 주문 금액이 큰 카테고리는 판매 수량도 많은가? | `completed` 주문, 2025-09-15 ~ 2026-09-14, 7개 카테고리 | 카테고리별 `total_quantity`, `total_sales`, `sales_ratio`, 수량 비중, 개당 평균 금액(= total_sales ÷ total_quantity) | 카테고리별 합계 = completed 원본 합계 |
| 월별 completed 주문 금액 변화는 주문 수와 평균 주문 금액(AOV) 중 무엇과 함께 움직이는가? | `completed` 주문, 13개월 (2025-09, 2026-09는 부분 월) | 월별 `total_sales`, 월별 고유 `order_id` 수(`order_count`), 월별 `avg_order_value` | 월별 합계 = completed 원본 합계, 첫·마지막 달 실제 날짜 범위 확인 |
| 구매 금액이 큰 고객은 반복 구매 고객인가, 고액 단발 고객인가? | `completed` 주문이 있는 고객 100명 | 고객별 `order_count`, `total_sales`, `avg_order_value` | 고객별 합계 = completed 원본 합계 |

### 왜 이 질문을 선택했는가?

- 세 질문 모두 현재 데이터의 컬럼만으로 계산할 수 있습니다. 필요한 컬럼(`quantity`, `line_total`, `order_status`, `category`, `customer_id`)이 모두 있고, 테이블 간 미매칭도 0건입니다.
- 공식 노트북 15절의 `next_question` 중 세 개를 계산 가능한 지표로 바꾼 것입니다.
- 첫 번째 질문은 기본 질문 "카테고리별 매출은 어떻게 다른가?"를 직접 수정했습니다. 금액만 보면 금액이 큰 이유가 수량인지 단가인지 구분할 수 없기 때문입니다.
- 상품(카테고리) → 시간(월) → 고객 순으로 서로 다른 분석 축을 하나씩 다룹니다.
- 세 집계 모두 같은 completed 원본 합계로 다시 돌아와야 하므로, 집계 실수를 바로 확인할 수 있습니다.

## 2. 핵심 집계 결과
- completed 원본 합계: 148,990,000원 (completed 주문 184건, 주문 상세 474행)
  - 병합 전후 주문 상세 행 수 764 → 764, orders와 연결되지 않은 행 0건
  - 참고: 상태 구분 없이 합산하면 255,610,000원 (cancelled 62,181,000원 + refunded 44,439,000원 포함)
- 카테고리별 합계: 148,990,000원
  - 스포츠 31,743,000 / 전자기기 26,400,000 / 생활용품 23,915,000 / 뷰티 23,383,000 / 식품 16,573,000 / 도서 16,389,000 / 패션 10,587,000
- 월별 합계: 148,990,000원 (13개월, 주문 184건)
- 총합 일치 여부: PASS(원본 = 카테고리별 = 월별 = 고객별 148,990,000원)

![핵심 EDA 결과](images/step02_eda.png)

### 결과 관찰
- 금액 1위는 스포츠(21.31%), 2위는 전자기기(17.72%)입니다. 상위 4개 카테고리(스포츠·전자기기·생활용품·뷰티)가 전체의 70.8%를 차지합니다.
- 카테고리별 금액 비중과 수량 비중의 차이는 ±3%p 이내입니다. 가장 큰 차이는 생활용품 −2.9%p, 식품 +1.9%p입니다.
- 월별 금액이 가장 큰 달은 2025-12(21,085,000원, 주문 23건)이고, 다음은 2026-03(18,323,000원, 주문 25건)입니다.
- 부분 월을 제외하면 2026-02(4,784,000원, 7건)와 2026-08(4,365,000원, 4건)이 가장 낮습니다.
- completed 주문이 있는 고객 100명 중 2회 이상 구매한 고객은 56명이고, 이들이 전체 금액의 74.8%를 차지합니다.
- completed Gate를 빼고 합산하면 카테고리 순위가 바뀝니다. 뷰티가 2위(47,551,000원)가 되지만, completed 기준으로는 4위(23,383,000원)입니다.

### 나의 해석과 판단
- 카테고리 간 금액 차이는 단가보다 판매 수량 차이와 함께 움직이는 것으로 관찰됩니다. 금액 비중과 수량 비중이 거의 같기 때문입니다. 
- 월별 금액은 주문 수와 같은 방향으로 움직이는 달이 많습니다.
- 2026-08의 AOV(1,091,250원)는 가장 높지만, 주문이 4건뿐이라 의미를 두기 어렵습니다.
- 주문 상태 필터 하나로 카테고리 순위가 바뀌므로, 금액성 분석은 completed Gate를 먼저 정한 뒤 시작해야 한다고 판단했습니다.
- 금액이 큰 고객 중에는 반복 구매형과 고액 소수 구매형이 섞여 있습니다. 그래서 `total_sales` 하나만으로 우수 고객을 정의하지 않습니다.

### 업무·분석적 의미
- 금액 보고서에는 집계 기준(주문 상태, 기간, 부분 월 여부)을 함께 적어야 서로 다른 보고서의 숫자를 비교할 수 있습니다.
- 여기서 `total_sales`는 completed 주문에 속한 `line_total` 합계입니다. 할인,배송비,세금,부분 환불을 반영한 회계상 순매출이 아니라는 점을 보고서에 밝혀야 합니다.
- 월별 금액 변화는 주문 수와 AOV로 나눠 봐야, 어느 쪽의 변화인지 구분할 수 있습니다.

### 한계와 추가 확인 사항
- completed 주문이 184건뿐이라, 월별·카테고리별 셀 하나가 주문 몇 건으로 이루어져 있습니다. 작은 차이는 우연일 수 있습니다.
- 2025-09(15일부터), 2026-09(14일까지)는 부분 월이라 다른 달과 직접 비교할 수 없습니다.
- 기간이 약 12개월이라 같은 달을 두 해 비교하는 계절성 분석은 할 수 없습니다.
- 주문 56건의 주문일이 고객 가입일보다 앞섭니다. 값을 바꾸지 않고 기록했으며, 원인은 Chapter 05 전처리 기준에서 확인해야 합니다.

## 3. 관찰 → 가설 → 추가 검증
### 사례 1
- 관찰: 상태 구분 없이 합산하면 뷰티가 금액 2위이지만, completed 기준으로는 4위입니다. 뷰티는 주문 금액 중 취소 금액 비중이 32.0%입니다.
- 가설: 뷰티 카테고리는 취소된 주문의 금액이 큰 편이라 completed 기준 순위가 낮아졌을 수 있습니다. 취소가 자주 일어나는지는 아직 모릅니다.
- 추가 검증:
  - 금액이 아닌 **주문 수 기준** 취소·환불률을 (주문, 카테고리) 단위로 비교하고, 카이제곱 검정을 했습니다.
  - 결과: cancelled 비율은 18.2%(스포츠) ~ 25.0%(식품) 범위이고, p-value = 0.969로 카테고리 간 차이는 우연으로 볼 수 있는 수준입니다.
  - 따라서 "뷰티는 취소가 많은 카테고리"라고 말하지 않습니다. 다음으로 취소된 주문과 completed 주문의 주문당 금액 크기를 비교합니다.

### 사례 2
- 관찰: 2026-02(4,784,000원)와 2026-08(4,365,000원)은 정상 월인데 금액이 가장 낮습니다. completed 주문 수도 7건, 4건으로 적습니다. 2026-08은 전체 주문 16건 중 completed가 4건입니다.
- 가설: 주문 수 자체의 감소와, 취소·환불 비중 증가가 함께 영향을 주었을 수 있습니다.
- 추가 검증: 월별 주문 상태 비율을 계산합니다. 또 월별 금액을 `order_count × avg_order_value`로 나눠, 어느 쪽이 더 크게 변했는지 비교합니다.

### 사례 3
- 관찰: AOV 상위 25%(1,061,750원 이상)인 1회 구매 고객은 15명이며, 금액 비중은 13.6%입니다. 2회 이상 구매한 고객 56명의 금액 비중은 74.8%입니다.
- 가설: 이 쇼핑몰의 completed 금액은 고액 단발 구매보다 반복 구매 고객과 더 크게 연결되어 있을 수 있습니다.
- 추가 검증:
  - 고객마다 가입 시점이 달라 재구매할 시간이 다릅니다. 가입 후 같은 기간(예: 180일) 안의 주문 수로 다시 세그먼트를 나눕니다.
  - 단, 가입일 이전 주문 56건의 처리 기준을 먼저 정해야 합니다.

## 4. LLM 질문 확장
- Safe Context: 
  - LLM에는 4개 테이블(customers, products, orders, order_items)의 테이블명과 컬럼명(스키마)만 제공했다.
  - 실제 행 데이터, 고객 식별 정보(customer_id 값, 성별·나이·도시 값), 매출 수치는 입력하지 않았다.
  - 데이터 규모(고객 150명, 결제수단별 주문 70~79건, 카테고리별 상품 7~19개, 데이터 기간 2025-09~2026-09)는 LLM 응답을 받은 뒤 직접 EDA로 확인하여 반영했다.

- Prompt: 현재 데이터로 검증 가능한 EDA 질문 10개를 만들고, 각 질문에 필요한 데이터셋과 pandas 기능을 정리하고, 계산 가능한 지표로 바꿔 달라
- LLM 제안 요약: 주문 상태(Q1), 월별 매출 추이(Q2), 카테고리 기여도(Q3), 결제수단별 취소·환불률(Q4), 요일 패턴(Q5), 장바구니 규모(Q6), 재구매(Q7), 고객 세그먼트별 1인당 매출(Q8), 매출 집중도·파레토(Q9), 가입→첫 주문 소요일 및 정합성 점검(Q10)의 10개 질문을 제안했다. 또한, 각 질문마다 계산식 형태의 지표, 필요한 테이블 조합(merge 경로), pandas 함수(value_counts, groupby/agg, crosstab, pct_change, pd.cut, cumsum 등)를 함께 제시했다.
- 실제 사용 여부: 수정 후 사용 
- 판단 이유: 
  - 10개 질문 모두 제공된 컬럼만으로 계산 가능하고, 마케팅 채널·유입 경로·재고처럼 데이터에 없는 변수를 끌어오지 않아 "추측 금지" 조건을 지켰다.
  - 다만 LLM은 실제 데이터를 보지 않았기 때문에 표본 크기, 기간 경계, 그룹 간 불균형을 고려하지 못했다. 매출 정의(어떤 주문 상태를 매출로 볼지)도 질문마다 일관되지 않아 보완이 필요했다.

- 사람이 수정한 내용:
  1. 매출 정의 통일: 매출이 들어가는 모든 지표(Q2·Q3·Q5·Q6·Q8·Q9)를 `order_status == 'completed'` 주문의 `line_total` 합으로 통일했다.
  2. 부분 월 처리(Q2):데이터를 확인해 2025-09와 2026-09가 부분 월임을 파악하고, 표시 또는 제외 조건을 추가했다.
  3. 카테고리 공정 비교(Q3): 카테고리별 상품 수가 7~19개로 차이 나는 것을 확인하고, 총매출 외에 "상품 1개당 매출" 지표를 추가했다.
  4. 집계 단위 명확화(Q5·Q6): 행(line) 단위가 아닌 주문 단위로 먼저 합계를 낸 뒤 평균을 구하도록 순서를 명시했다. 요일은 월~일 순서로 `reindex`하도록 했다.
  5. 재구매율 분모·분자 정의(Q7): 전환 고객 비율(126/150, 실제 데이터로 확인)과 재구매율(completed 2회 이상 ÷ completed 1회 이상)을 분리해 정의했다. 주문 없는 고객은 `merge(how='left')` 후 `fillna(0)`으로 포함했다.
  6. 정합성 점검 추가(Q10): 가입일 이전 주문(음수 일수)의 건수와 비율을 함께 보도록 지표를 확장했다.


![LLM EDA 질문 확장](images/step04_llm.png)

## 5. 다음 분석 우선순위
1. 취소·환불된 주문의 주문당 금액은 completed 주문보다 큰가?
2. 2026-02, 2026-08의 금액 저하는 주문 수 감소와 취소·환불 증가 중 무엇이 더 큰가?
3. 가입일보다 앞선 주문 56건은 데이터 오류인가, 가입일 정의의 차이인가?

### 가장 먼저 확인하고 싶은 이유
- 주문 수 기준 취소율은 카테고리 간 차이가 없었습니다(p = 0.969). 그런데 금액 기준 취소 비중은 15.1%(도서) ~ 33.5%(패션)로 차이가 큽니다.
- 이 차이를 설명할 다음 단서가 취소된 주문의 금액 크기입니다.
- 결과에 따라 completed Gate 적용 후 카테고리 순위가 바뀐 이유를 설명할 수 있고, 어떤 카테고리를 더 자세히 볼지 판단할 수 있습니다.

### 현재 데이터만으로 단정할 수 없는 것
- 주문이 취소·환불된 이유 
- 카테고리별 이익·마진
- 할인·프로모션 효과
- 방문 대비 구매 전환율 (방문·세션 기록 없음)
- 회계상 순매출 


### 1. 내가 만든 질문과 지표
![내가 만든 질문과 지표](images/step01_question_metric.png)

- 관찰: 6개 영역별 질문·지표 정리
- 해석: 질문마다 지표·데이터가 연결됨
- 한계: 기본 질문이라 분석 범위 미정
- 다음 질문: completed 기준을 질문에 추가

### 2. 고객·상품·주문 기본 분포
![고객·상품·주문 기본 분포](images/step02_basic_distribution.png)

- 관찰: 성남 21명 최다, 고양 11명 최소
- 해석: 도시별 고객 수는 고른 편(7~14%)
- 한계: 고객 수일 뿐 구매 금액은 모름
- 다음 질문: 도시별 completed 금액 차이는?

### 3. completed Gate와 분석 범위
![completed Gate와 분석 범위](images/step03_completed_gate.png)

- 관찰: 병합 전후 764행, 누락 0건
- 해석: 상품 연결 시 행 증감·누락 없음
- 한계: completed 필터는 미적용 상태
- 다음 질문: completed 행·주문 수는 몇 개?

### 4. 카테고리·월·고객 핵심 집계
![카테고리·월·고객 핵심 집계](images/step04_group_analysis.png)

- 관찰: completed 기준 스포츠 21.3% 1위
- 해석: 금액·수량 비중이 거의 일치
- 한계: 주문 184건, 월별 표본이 작음
- 다음 질문: 반복 구매 고객 비중은 얼마?

### 5. source total과 그룹 합계 검증
![source total과 그룹 합계 검증](images/step05_validation.png)

- 관찰: 4개 합계 모두 148,990,000원
- 해석: 같은 범위로 집계, 누락 없음
- 한계: 할인·환불 반영 순매출 아님
- 다음 질문: Gate 전 합계와 차이 원인은?

### 6. LLM 질문 확장
![LLM EDA 질문 확장](images/step06_my_question.png)

- 관찰: LLM이 검증 가능 질문 10개 제안
- 해석: 질문 후보를 지표로 구체화함
- 한계: 제안 내용 정확성은 직접 검증
- 다음 질문: 제안 중 1개 골라 직접 분석

## 최종 체크
- [0] 질문과 지표가 연결되어 있습니다.
- [0] 집계 총합을 검증했습니다.
- [0] 관찰과 가설을 구분했습니다.
- [0] 다음 검증 질문을 작성했습니다.
- [0] LLM 제안을 검증했습니다.
- [0] 최종 Notebook URL을 제출합니다.



# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-study\notebooks\ch06
프로젝트 루트: c:\dev\llm-data-analysis-study\notebooks\ch06
전처리 데이터 폴더: c:\dev\llm-data-analysis-study\notebooks\ch06\data\processed
보고서 폴더: c:\dev\llm-data-analysis-study\notebooks\ch06\reports


관찰

현재 실행 위치와 프로젝트 루트가 모두 notebooks\ch06으로 출력되었습니다.
그 결과 PROCESSED_DIR는 notebooks\ch06\data\processed, REPORT_DIR는 notebooks\ch06\reports로 잡혔습니다.

가설

코드는 폴더 이름이 notebooks일 때만 상위 폴더로 이동합니다. 실제 실행 폴더 이름은 ch06이라 조건이 맞지 않아 루트를 잘못 잡았을 수 있습니다.

추가 검증

PROCESSED_DIR.exists()로 이 경로에 데이터가 실제로 있는지 확인합니다.
이후 셀에서 REPORT_DIR가 다시 정의되는지 확인합니다.

판단

5절의 find_project_root()(scripts 폴더 기준)로 루트를 한 번만 정하고, REPORT_DIR도 그 루트 기준으로 다시 정의하기로 했습니다.

## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [6]:
from pathlib import Path
import pandas as pd

# 프로젝트 루트 찾기 (scripts 폴더가 있는 곳)
def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / 'scripts').is_dir():
            return p
    raise RuntimeError('프로젝트 루트를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
print('PROCESSED_DIR =', PROCESSED_DIR)

required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


PROCESSED_DIR = c:\dev\llm-data-analysis-study\data\processed
전처리 데이터 불러오기 완료


관찰

find_project_root()로 찾은 PROCESSED_DIR는 c:\dev\llm-data-analysis-study\data\processed이고, 4개 파일을 모두 불러왔습니다.
4절의 경로(notebooks\ch06\...)와 다릅니다.

가설

이 셀에서 PROJECT_ROOT와 PROCESSED_DIR만 다시 정의되고 REPORT_DIR는 4절 값이 그대로 남아 있을 수 있습니다.

추가 검증

이 셀 직후 print(REPORT_DIR)로 보고서 저장 위치를 확인합니다.

판단

이 셀에 REPORT_DIR = PROJECT_ROOT / 'reports'를 추가해 데이터와 보고서 경로가 같은 루트를 쓰도록 수정하기로 했습니다.

## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [7]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (100, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (764, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



관찰

customers 150행, products 100행, orders 300행, order_items 764행입니다.
주문 1건당 평균 주문 상세 행은 764 ÷ 300 ≈ 2.55개입니다.
customers에는 name 컬럼이 있습니다.

가설

주문 1건에 여러 상품이 들어 있으므로, 행 단위로 평균을 내면 주문 단위 평균과 결과가 달라질 수 있습니다.

추가 검증

order_items.groupby('order_id').size().describe()로 주문당 행 수 분포를 확인합니다.

판단

금액 평균(AOV)은 항상 order_id별로 먼저 합계를 낸 뒤 계산하기로 했습니다. name은 분석과 저장에서 제외합니다.

In [8]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str)

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 0
주문 시작일: 2025-09-15 00:00:00
주문 종료일: 2026-09-14 00:00:00


관찰

order_date 변환 실패는 0건입니다.
주문 기간은 2025-09-15 ~ 2026-09-14입니다.

가설

데이터가 월 단위가 아니라 날짜 단위로 잘려 있어, 첫 달과 마지막 달은 다른 달보다 일수가 적을 수 있습니다.

추가 검증

2025-09(16일)와 2026-09(14일)의 실제 일수를 세고, 일평균 주문 수로 다른 달과 비교합니다.

판단

월별 비교에서 2025-09와 2026-09는 "부분 월"로 표시하고, 최고·최저 월 판단에서는 제외하기로 했습니다.

## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [9]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


관찰

고객·상품·주문·매출·시간·고객 가치 6개 영역에 질문 1개씩, 지표와 필요한 데이터가 연결되어 있습니다.
매출 질문(3~5번)에는 주문 상태 기준이 적혀 있지 않습니다.

가설

주문 상태 기준 없이 매출을 집계하면 취소·환불 금액이 매출에 섞일 수 있습니다.

추가 검증

10절의 주문 상태 분포에서 completed가 아닌 주문 비중을 확인합니다.

판단

매출이 들어가는 질문에는 모두 "completed 주문 기준"을 분석 범위로 추가하기로 했습니다.

## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [10]:
customers['age'].describe()


count    150.000000
mean      42.086667
std       15.613166
min       19.000000
25%       29.000000
50%       40.000000
75%       57.000000
max       69.000000
Name: age, dtype: float64

관찰

150명, 평균 42.1세, 중앙값 40세, 최소 19세, 최대 69세, 표준편차 15.6입니다.
1사분위 29세, 3사분위 57세로 IQR이 28세입니다.

가설

평균과 중앙값이 비슷하고 범위 전체에 넓게 퍼져 있어, 특정 연령대에 몰리지 않은 분포일 수 있습니다.

추가 검증

pd.cut으로 연령대(20대 이하~60대)를 나눠 연령대별 고객 수를 셉니다.

판단

연령대 구간표를 먼저 만들고, 연령대별 completed 1인당 구매 금액 비교로 이어 가기로 했습니다.

In [11]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,성남,21,14.00
1,광주,17,11.33
2,부산,16,10.67
3,대구,15,10.00
4,서울,15,10.00
5,울산,14,9.33
6,인천,14,9.33
7,대전,14,9.33
8,수원,13,8.67
9,고양,11,7.33


관찰

성남이 21명(14.0%)으로 가장 많고, 고양이 11명(7.3%)으로 가장 적습니다.
10개 도시가 7.3~14.0% 범위에 있습니다.

가설

도시별 고객 수 차이가 작아서, 도시별 구매 금액 차이는 고객 수보다 고객 1명당 구매 금액에서 생길 수 있습니다.

추가 검증

도시별 completed 구매 금액과 1인당 구매 금액을 계산합니다(22절 과제 1).

판단

고객 수가 아니라 구매 고객 1인당 금액으로 도시를 비교하기로 했습니다.

In [12]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,F,84,56.0
1,M,66,44.0


관찰

여성 84명(56.0%), 남성 66명(44.0%)입니다. 결측은 없습니다.

가설

성별 고객 수 차이가 성별 매출 차이로 그대로 이어질 수도 있고, 1인당 구매 금액이 달라 매출 비중은 다르게 나올 수도 있습니다.

추가 검증

성별 completed 매출 비중과 1인당 매출을 고객 수 비중(56:44)과 비교합니다.

판단

성별은 연령대와 함께 세그먼트(성별 × 연령대)로 묶어 1인당 매출을 보기로 했습니다.

고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가? 성남(21명, 14.0%)이 가장 많고, 광주(11.3%)·부산(10.7%)이 뒤를 잇는다.
    다만 10개 도시가 7~14% 범위에 고르게 분포해 특정 도시 쏠림은 크지 않다.
- 고객의 평균 나이는 어느 정도인가? 평균 42.1세, 중앙값 40세. 19~69세로 연령대가 넓게 퍼져 있다(표준편차 15.6).
- 성별 고객 수는 어떻게 분포하는가? 여성 84명(56%), 남성 66명(44%)으로 여성 고객이 다소 많다
- 도시별 구매 금액도 차이가 있는가? 위의 테이블에서는 확인 불가, 추가 작업이 필요하다.


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [13]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,스포츠,19,19.0
1,전자기기,17,17.0
2,생활용품,16,16.0
3,뷰티,16,16.0
4,도서,14,14.0
5,패션,11,11.0
6,식품,7,7.0


관찰

스포츠 19개가 가장 많고, 식품 7개가 가장 적습니다. 최다와 최소가 약 2.7배 차이입니다.

가설

상품 수가 많은 카테고리는 선택지가 많아 판매 수량과 매출도 클 수 있습니다.

추가 검증

카테고리별 상품 수와 completed 매출의 상관을 계산하고, 상품 1개당 매출을 구합니다.

판단

카테고리 총매출을 비교할 때 상품 1개당 매출을 함께 보기로 했습니다.

In [14]:
products['price'].describe()


count       100.000000
mean     110040.000000
std       56433.910574
min        5000.000000
25%       65750.000000
50%      112000.000000
75%      161000.000000
max      200000.000000
Name: price, dtype: float64

관찰

상품 100개 가격은 5,000원 ~ 200,000원, 평균 110,040원, 중앙값 112,000원, 표준편차 56,434원입니다.

가설

평균과 중앙값이 거의 같고 범위 전체에 고르게 퍼져 있어, 극단적인 고가 상품이 매출을 끌어올리는 구조는 아닐 수 있습니다.

추가 검증

가격 히스토그램을 그리고, 가격 구간별 completed 판매 수량을 계산합니다.

판단

가격 구간(pd.cut(price))별 판매 수량 비교를 7장 시각화 후보로 두기로 했습니다.

In [15]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
4,식품,7,137143.0,67000,197000
1,뷰티,16,117688.0,5000,200000
6,패션,11,115909.0,28000,198000
3,스포츠,19,111579.0,10000,196000
0,도서,14,106857.0,25000,174000
5,전자기기,17,101588.0,5000,165000
2,생활용품,16,96438.0,11000,188000


관찰

평균 가격은 식품이 137,143원으로 가장 높고, 생활용품이 96,438원으로 가장 낮습니다.
최저가 5,000원 상품은 뷰티와 전자기기에 있습니다.
식품은 상품 수가 가장 적고(7개), 최저가도 67,000원으로 가장 높습니다.

가설

식품은 상품 수가 적어도 단가가 높아 개당 매출이 클 수 있습니다.

추가 검증

이 평균은 상품 목록 가격의 평균입니다. 실제로 팔린 상품의 개당 금액(total_sales ÷ total_quantity)과 비교합니다.

판단

카테고리 단가는 목록 평균이 아니라 실제 판매 기준 개당 금액으로 보기로 했습니다(12절 [47]의 sales_per_unit).

## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [16]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,184,61.33
1,cancelled,64,21.33
2,refunded,52,17.33


관찰

전체 300건 중 completed 184건(61.3%), cancelled 64건(21.3%), refunded 52건(17.3%)입니다.
주문 10건 중 약 4건이 매출로 볼 수 없는 주문입니다.

가설

상태 구분 없이 매출을 합산하면 매출이 크게 부풀려지고, 카테고리·월 순위도 달라질 수 있습니다.

추가 검증

같은 집계를 전체 주문 기준과 completed 기준으로 각각 계산해 순위를 비교합니다.

판단

금액성 분석은 모두 order_status == 'completed' Gate를 먼저 적용한 뒤 시작하기로 했습니다.

In [17]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,79,26.33
1,naver_pay,77,25.67
2,bank_transfer,74,24.67
3,card,70,23.33


관찰

카카오페이 79건(26.3%), 네이버페이 77건(25.7%), 계좌이체 74건(24.7%), 카드 70건(23.3%)입니다. 차이는 최대 9건입니다.

가설

결제수단별 주문 수가 비슷하므로, 결제수단별 차이는 주문 수보다 취소·환불률이나 주문 금액에서 나타날 수 있습니다.

추가 검증

pd.crosstab(payment_method, order_status, normalize='index')로 결제수단별 상태 비율을 계산하고 카이제곱 검정을 합니다.

판단

결제수단은 주문 수가 아니라 취소·환불률과 completed 평균 주문 금액으로 비교하기로 했습니다.

In [18]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-09-15 00:00:00
주문 종료일: 2026-09-14 00:00:00
주문 월 개수: 13


관찰

주문 기간은 2025-09-15 ~ 2026-09-14이고, 주문 월은 13개입니다.

가설

기간이 약 12개월이라 같은 달을 두 해 비교할 수 없으므로, 월별 차이를 계절성으로 설명하기는 어렵습니다.

추가 검증

같은 달(9월)이 두 번 나오지만 둘 다 부분 월입니다. 9월끼리 일평균 기준으로만 비교 가능한지 확인합니다.

판단

월별 결과는 "추세" 또는 "계절성"이라는 표현을 쓰지 않고 "월별 변동"으로만 기록하기로 했습니다.

## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [19]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,3,102000,306000
1,5,25000,125000
2,3,142000,426000
3,3,193000,579000
4,4,189000,756000


관찰

상위 5행에서 line_total = quantity × unit_price가 모두 맞습니다(예: 3 × 102,000 = 306,000).

가설

전처리 단계에서 계산된 값이라 764행 전체에서 같은 규칙이 성립할 수 있습니다. 다만 5행만 확인했습니다.

추가 검증

(order_items['quantity'] * order_items['unit_price'] != order_items['line_total']).sum()으로 전체 행을 확인합니다.

판단

전체 행 검증 코드를 이 셀 아래 한 줄 추가하기로 했습니다.

In [20]:
sales_items = order_items.merge(
    products,
    on='product_id',
    how='left',
)

print('병합 전 order_items:', order_items.shape)
print('병합 후 sales_items:', sales_items.shape)
print('상품명 누락:', sales_items['product_name'].isna().sum())
print('카테고리 누락:', sales_items['category'].isna().sum())

sales_items.head()


병합 전 order_items: (764, 6)
병합 후 sales_items: (764, 9)
상품명 누락: 0
카테고리 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price
0,1,1,100,3,102000,306000,도서 상품 100,도서,102000
1,2,1,87,5,25000,125000,도서 상품 087,도서,25000
2,3,1,7,3,142000,426000,도서 상품 007,도서,142000
3,4,1,9,3,193000,579000,스포츠 상품 009,스포츠,193000
4,5,2,72,4,189000,756000,뷰티 상품 072,뷰티,189000


관찰

병합 전후 행 수가 764 → 764로 같고, 상품명·카테고리 누락은 0건입니다.
상위 5행에서는 unit_price와 price가 같습니다.

가설

판매 단가가 목록 가격과 항상 같다면, 데이터에 할인 정보가 없다고 볼 수 있습니다.

추가 검증

(sales_items['unit_price'] != sales_items['price']).sum()을 계산합니다.

판단

결과가 0이면 보고서에 "할인 정보 없음(unit_price = price)"을 분석 한계로 명시하기로 했습니다.

## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [47]:
# ===== STEP 5. 카테고리·월·고객 핵심 집계 (completed Gate 기준) =====
import pandas as pd

# completed Gate: 주문 상세 + 주문 정보 연결 후 completed만 남김
completed_items = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]], on="order_id", how="left"
)
completed_items = completed_items[completed_items["order_status"] == "completed"].copy()
completed_items["order_month"] = pd.to_datetime(completed_items["order_date"]).dt.to_period("M").astype(str)
completed_items = (completed_items
                   .merge(products[["product_id", "category"]], on="product_id", how="left")
                   .merge(customers[["customer_id", "city"]], on="customer_id", how="left"))
print(f"분석 범위: completed 주문 {completed_items['order_id'].nunique()}건 / "
      f"합계 {completed_items['line_total'].sum():,}원\n")

# 1) 카테고리별
category_sales = (completed_items
                  .groupby("category", as_index=False)
                  .agg(total_quantity=("quantity", "sum"), total_sales=("line_total", "sum"))
                  .sort_values("total_sales", ascending=False))
category_sales["quantity_ratio"] = (category_sales["total_quantity"] / category_sales["total_quantity"].sum() * 100).round(2)
category_sales["sales_ratio"] = (category_sales["total_sales"] / category_sales["total_sales"].sum() * 100).round(2)
category_sales["sales_per_unit"] = (category_sales["total_sales"] / category_sales["total_quantity"]).round(0)

# 2) 월별 (order_count = 고유 order_id 수)
monthly_sales = (completed_items
                 .groupby("order_month", as_index=False)
                 .agg(total_sales=("line_total", "sum"), order_count=("order_id", "nunique"))
                 .sort_values("order_month"))
monthly_sales["avg_order_value"] = (monthly_sales["total_sales"] / monthly_sales["order_count"]).round(0)
# 부분 월 판단은 전체 주문 기간 기준 (데이터 수집 범위)
first_day = pd.to_datetime(orders["order_date"]).min()
last_day = pd.to_datetime(orders["order_date"]).max()
monthly_sales["note"] = ""
monthly_sales.loc[monthly_sales["order_month"] == first_day.strftime("%Y-%m"), "note"] = f"부분 월({first_day:%m-%d}~)"
monthly_sales.loc[monthly_sales["order_month"] == last_day.strftime("%Y-%m"), "note"] = f"부분 월(~{last_day:%m-%d})"

# 3) 고객별 (개인정보 보호: 이름 제외, customer_id와 city만 사용)
customer_sales = (completed_items
                  .groupby(["customer_id", "city"], as_index=False)
                  .agg(order_count=("order_id", "nunique"), total_sales=("line_total", "sum"))
                  .sort_values("total_sales", ascending=False))
customer_sales["avg_order_value"] = (customer_sales["total_sales"] / customer_sales["order_count"]).round(0)

money = "{:,.0f}"
print("[1] 카테고리별 집계")
display(category_sales.style.format({"total_quantity": "{:,}", "total_sales": money, "quantity_ratio": "{:.2f}",
                                     "sales_ratio": "{:.2f}", "sales_per_unit": money}).hide(axis="index"))
print("[2] 월별 집계")
display(monthly_sales.style.format({"total_sales": money, "avg_order_value": money}).hide(axis="index"))
print(f"[3] 고객별 집계 (상위 10명 / 전체 {len(customer_sales)}명)")
display(customer_sales.head(10).style.format({"total_sales": money, "avg_order_value": money}).hide(axis="index"))

분석 범위: completed 주문 184건 / 합계 148,990,000원

[1] 카테고리별 집계


category,total_quantity,total_sales,quantity_ratio,sales_ratio,sales_per_unit
스포츠,295,"31,743,000",20.46,21.31,"107,603"
전자기기,259,"26,400,000",17.96,17.72,"101,931"
생활용품,272,"23,915,000",18.86,16.05,"87,923"
뷰티,223,"23,383,000",15.46,15.69,"104,857"
식품,133,"16,573,000",9.22,11.12,"124,609"
도서,149,"16,389,000",10.33,11.00,"109,993"
패션,111,"10,587,000",7.70,7.11,"95,378"


[2] 월별 집계


order_month,total_sales,order_count,avg_order_value,note
2025-09,"2,650,000",5,"530,000",부분 월(09-15~)
2025-10,"16,970,000",19,"893,158",
2025-11,"11,021,000",13,"847,769",
2025-12,"21,085,000",23,"916,739",
2026-01,"14,272,000",16,"892,000",
2026-02,"4,784,000",7,"683,429",
2026-03,"18,323,000",25,"732,920",
2026-04,"11,546,000",16,"721,625",
2026-05,"16,631,000",19,"875,316",
2026-06,"12,504,000",17,"735,529",


[3] 고객별 집계 (상위 10명 / 전체 100명)


customer_id,city,order_count,total_sales,avg_order_value
117,성남,5,"4,100,000","820,000"
102,고양,4,"3,996,000","999,000"
83,수원,4,"3,880,000","970,000"
30,서울,5,"3,590,000","718,000"
40,서울,4,"3,523,000","880,750"
20,인천,2,"3,191,000","1,595,500"
3,성남,2,"3,178,000","1,589,000"
111,광주,3,"3,153,000","1,051,000"
66,서울,4,"3,093,000","773,250"
147,부산,2,"2,990,000","1,495,000"


관찰

completed 주문 184건, 합계 148,990,000원입니다.
카테고리: 스포츠 21.31%가 1위입니다. 금액 비중과 수량 비중 차이는 ±3%p 이내이고, 가장 큰 차이는 생활용품(수량 18.86% vs 금액 16.05%)과 식품(9.22% vs 11.12%)입니다. 개당 금액은 식품 124,609원이 가장 높고, 생활용품 87,923원이 가장 낮습니다.
월: 부분 월을 제외하면 2025-12가 21,085,000원(23건)으로 가장 높습니다. 2026-02(4,784,000원, 7건)와 2026-08(4,365,000원, 4건)이 가장 낮습니다. 2026-08의 AOV는 1,091,250원으로 가장 높습니다.
고객: completed 구매 고객은 100명입니다. 상위 10명 중 7명은 3~5회 구매했고, 3명(20, 3, 147번)은 2회 구매에 AOV가 약 150만 원입니다.

가설

카테고리 매출 차이는 주로 판매 수량 차이에서 오지만, 식품(단가 높음)과 생활용품(단가 낮음)에서는 단가도 일부 영향을 줄 수 있습니다.
월별 금액은 주문 수와 같은 방향으로 움직이며, 2026-08의 높은 AOV는 주문 4건에서 나온 우연일 수 있습니다.

추가 검증

카테고리별로 sales_ratio − quantity_ratio를 계산해, 그 차이가 sales_per_unit 순위와 일치하는지 확인합니다.
월별 total_sales와 order_count, total_sales와 avg_order_value의 상관을 각각 계산합니다(부분 월 제외 11개월).

판단

월별 매출은 주문 수와 AOV 중 무엇과 함께 움직이는지 상관 2개를 먼저 계산하기로 했습니다. 내 EDA 질문 2번에 바로 답할 수 있기 때문입니다.

In [21]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14
0,도서,238,24645000,9.64
6,패션,220,23801000,9.31


관찰

상태 구분 없이 합산하면 합계 255,610,000원, 수량 2,333개입니다. 뷰티가 47,551,000원으로 2위입니다.
completed 기준([47])과 비교하면 completed가 아닌 금액 비중은 패션 55.5%, 뷰티 50.8%, 식품 50.7%로 높고, 생활용품 31.4%로 가장 낮습니다.
뷰티는 순위가 2위 → 4위로 내려갑니다.

가설

패션·뷰티·식품은 취소·환불 주문의 금액이 커서 순위 변화가 생겼을 수 있습니다.
주문 수 기준 취소율은 카테고리 간 차이가 없었으므로(p = 0.969), 차이는 "얼마나 자주"보다 "취소된 주문이 얼마나 큰가"에서 올 수 있습니다.

추가 검증

카테고리별로 completed 주문과 cancelled·refunded 주문의 주문당 금액(평균·중앙값)을 비교합니다.

판단

이 셀의 표는 비교용으로만 남기고, 결과 보고에는 completed 기준 표만 사용하기로 했습니다.

In [22]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,스포츠,468,50174000,19.63,111579.0
1,뷰티,376,47551000,18.60,117688.0
2,전자기기,401,41003000,16.04,101588.0
3,생활용품,390,34839000,13.63,96438.0
4,식품,240,33597000,13.14,137143.0
5,도서,238,24645000,9.64,106857.0
6,패션,220,23801000,9.31,115909.0


관찰

전체 상태 기준 매출에 상품 목록 평균 가격을 붙인 표입니다.
목록 평균 가격 1위인 식품(137,143원)은 매출 5위, 가격 최하위 생활용품(96,438원)은 매출 4위입니다.

가설

목록 평균 가격은 실제로 팔린 상품의 구성을 반영하지 않으므로, 단가 효과를 과대 또는 과소 평가할 수 있습니다.
또한 이 표의 매출은 completed Gate 전 값이라 비교 기준이 섞여 있습니다.

추가 검증

completed 기준 sales_per_unit(실제 개당 금액)과 목록 평균 가격의 차이를 카테고리별로 계산합니다.

판단

이 표 대신 [47]의 completed 기준 sales_per_unit을 단가 지표로 쓰기로 했습니다.

해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [23]:
order_sales = order_items.merge(
    orders,
    on='order_id',
    how='left',
)

order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str)

print('병합 후 order_sales:', order_sales.shape)
print('order_date 누락:', order_sales['order_date'].isna().sum())

order_sales.head()


병합 후 order_sales: (764, 12)
order_date 누락: 0


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,100,3,102000,306000,123,2026-07-14,card,completed,2026-07,Tuesday
1,2,1,87,5,25000,125000,123,2026-07-14,card,completed,2026-07,Tuesday
2,3,1,7,3,142000,426000,123,2026-07-14,card,completed,2026-07,Tuesday
3,4,1,9,3,193000,579000,123,2026-07-14,card,completed,2026-07,Tuesday
4,5,2,72,4,189000,756000,77,2025-09-29,naver_pay,cancelled,2025-09,Monday


관찰

병합 후 764행, 12개 컬럼이고 order_date 누락은 0건입니다.
첫 5행에 completed 주문과 cancelled 주문이 함께 들어 있습니다.

가설

이 병합 결과를 그대로 월별 집계에 쓰면 취소 주문 금액이 월 매출에 포함될 수 있습니다.

추가 검증

order_sales['order_status'].value_counts()로 행 단위 상태 분포를 확인합니다(completed 474행 예상).

판단

월별 집계는 이 표에 completed 필터를 적용한 뒤 계산하기로 했습니다.

In [24]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-09,4950000,9,550000.0
1,2025-10,25481000,27,943741.0
2,2025-11,18572000,22,844182.0
3,2025-12,30112000,33,912485.0
4,2026-01,29110000,29,1003793.0
5,2026-02,14289000,17,840529.0
6,2026-03,25161000,35,718886.0
7,2026-04,24160000,27,894815.0
8,2026-05,25061000,28,895036.0
9,2026-06,18316000,25,732640.0


관찰

전체 주문 300건 기준 월별 집계입니다.
월별 completed 비율([47] completed 주문 수 ÷ 이 표 주문 수)은 2026-08이 4/16 = 25.0%, 2026-02가 7/17 = 41.2%로 가장 낮습니다. 부분 월을 제외한 평균은 174/283 = 61.5%입니다.
부분 월을 제외한 월평균 전체 주문 수는 약 25.7건인데, 2026-02는 17건, 2026-08은 16건입니다.
2026-01은 전체 기준 29,110,000원이지만 completed 기준은 14,272,000원(49.0%)입니다.

가설

2026-02와 2026-08의 금액 저하는 주문 수 감소와 completed 비율 하락이 함께 영향을 주었을 수 있습니다.

추가 검증

completed 주문 수 = 전체 주문 수 × completed 비율로 나눕니다.
각 달에 대해 "주문 수만 평균이었다면"과 "completed 비율만 평균이었다면"의 completed 주문 수를 계산해, 어느 요인이 더 큰지 비교합니다.

판단

사례 2 가설 검증으로 이 주문 수 × completed 비율 분해를 먼저 하기로 했습니다. 이미 있는 두 표로 계산할 수 있기 때문입니다.

월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [46]:
# ===== STEP 6. 집계 총합 검증 (completed Gate 기준) =====
import pandas as pd

# 1) completed Gate: 주문 상세 + 주문 정보 연결 후 completed만 남김
gate = order_items.merge(orders[["order_id", "customer_id", "order_date", "order_status"]],
                         on="order_id", how="left")
completed_items = gate[gate["order_status"] == "completed"].copy()
completed_items["order_month"] = pd.to_datetime(completed_items["order_date"]).dt.to_period("M").astype(str)
completed_items = (completed_items
                   .merge(products[["product_id", "category"]], on="product_id", how="left")
                   .merge(customers[["customer_id", "city"]], on="customer_id", how="left"))

# 2) 같은 completed 범위로 카테고리·월·고객별 집계
category_total = completed_items.groupby("category", dropna=False)["line_total"].sum()
monthly_total  = completed_items.groupby("order_month", dropna=False)["line_total"].sum()
customer_total = completed_items.groupby(["customer_id", "city"], dropna=False)["line_total"].sum()

# 3) source total과 그룹별 합계 비교
source_total = completed_items["line_total"].sum()

validation = pd.DataFrame({
    "구분": ["completed 원본 합계", "카테고리별 합계", "월별 합계", "고객별 합계"],
    "집계 단위": [f'{len(completed_items)}행 / {completed_items["order_id"].nunique()}주문',
                 f"{category_total.size}개 카테고리", f"{monthly_total.size}개 월", f"{customer_total.size}명 고객"],
    "합계(원)": [source_total, category_total.sum(), monthly_total.sum(), customer_total.sum()],
})
validation["원본과 차이(원)"] = validation["합계(원)"] - source_total
validation["판정"] = ["기준"] + ["PASS" if d == 0 else "CHECK" for d in validation["원본과 차이(원)"][1:]]

print("분석 범위: order_status == 'completed'")
print(f"completed 주문 기간: {pd.to_datetime(completed_items['order_date']).min():%Y-%m-%d} ~ "
      f"{pd.to_datetime(completed_items['order_date']).max():%Y-%m-%d}")
display(validation.style.format({"합계(원)": "{:,}", "원본과 차이(원)": "{:,}"}).hide(axis="index"))

overall = "PASS" if (validation["원본과 차이(원)"] == 0).all() else "CHECK"
print("최종 일치 여부:", overall)

분석 범위: order_status == 'completed'
completed 주문 기간: 2025-09-15 ~ 2026-09-11


구분,집계 단위,합계(원),원본과 차이(원),판정
completed 원본 합계,474행 / 184주문,"148,990,000",0,기준
카테고리별 합계,7개 카테고리,"148,990,000",0,PASS
월별 합계,13개 월,"148,990,000",0,PASS
고객별 합계,100명 고객,"148,990,000",0,PASS


최종 일치 여부: PASS


관찰

상태 구분 없이 집계하면 1위는 102번 고객(5,977,000원, 5회)입니다. completed 기준으로는 3,996,000원(4회)으로 2위입니다.
142번 고객은 전체 기준 4위(4,862,000원)이지만 completed 기준 상위 10명에 없습니다.
이 표에는 고객 이름(name)이 들어 있습니다.

가설

일부 고객은 취소·환불 금액이 커서, 상태 기준에 따라 "우수 고객" 명단이 달라질 수 있습니다.

추가 검증

고객별로 completed 금액과 전체 금액의 차이를 계산하고, 차이가 큰 고객 수를 셉니다.

판단

고객 분석은 completed 기준으로만 하고, 이 표와 저장 파일에서 name 컬럼을 제거하기로 했습니다. 저장소가 GitHub에 공개되기 때문입니다.

In [25]:
customer_sales_base = order_sales.merge(
    customers,
    on='customer_id',
    how='left',
)

group_columns = ['customer_id', 'city']
if 'name' in customer_sales_base.columns:
    group_columns = ['customer_id', 'name', 'city']

customer_sales = (
    customer_sales_base
    .groupby(group_columns, as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales['avg_order_value'] = (
    customer_sales['total_sales'] / customer_sales['order_count']
).round(0)

customer_sales.head(10)


,customer_id,name,city,order_count,total_sales,avg_order_value
79,102,이진호,고양,5,5977000,1195400.0
93,117,김지원,성남,6,5552000,925333.0
67,83,전은경,수원,5,4951000,990200.0
118,142,하서연,서울,3,4862000,1620667.0
34,40,박예준,서울,5,4857000,971400.0
24,30,이민재,서울,7,4853000,693286.0
0,3,이경수,성남,3,4668000,1556000.0
10,14,윤성호,대전,3,4267000,1422333.0
122,146,김숙자,성남,3,4108000,1369333.0
87,111,류정훈,광주,4,4067000,1016750.0


## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [26]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


관찰

기본 질문 5개마다 결과표와 다음 질문이 1개씩 연결되어 있습니다.
다음 질문 중 "수량 때문인가 단가 때문인가", "고액 구매 고객은 반복 구매 고객인가"는 이미 [47]과 [45]에서 답했습니다.

가설

이 표는 completed Gate 적용 전에 만들어져, 질문에 분석 범위가 빠져 있을 수 있습니다.

추가 검증

각 질문이 completed 기준 결과표와 연결되는지 다시 확인합니다.

판단

next_question에 "(completed 기준)"을 붙이고, 이미 답한 질문에는 결과 요약 한 줄을 추가하기로 했습니다.

## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [27]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch06/reports/ch06_category_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch06/reports/ch06_customer_city.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch06/reports/ch06_customer_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch06/reports/ch06_eda_questions.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch06/reports/ch06_monthly_sales.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch06/reports/ch06_product_category.csv')]

관찰

CSV 6개가 notebooks\ch06\reports에 저장되었습니다. 프로젝트 루트의 reports가 아닙니다.
이 셀은 [27]에 실행되었고, completed 기준 집계 셀([46], [47])은 그 이후에 실행되었습니다.

가설

저장된 ch06_category_sales.csv, ch06_monthly_sales.csv, ch06_customer_sales.csv는 completed Gate 전(전체 상태 기준) 값이고, 고객 파일에는 이름이 포함되어 있을 수 있습니다.

추가 검증

저장된 ch06_category_sales.csv를 다시 읽어 합계가 255,610,000원인지 148,990,000원인지 확인합니다.

판단

REPORT_DIR를 프로젝트 루트 기준으로 고친 뒤, completed 기준 표(이름 제외)로 다시 저장하기로 했습니다. 7장 시각화가 이 파일을 쓰기 때문입니다.

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [28]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: c:\dev\llm-data-analysis-study\notebooks\ch06\reports\ch06_eda_summary.md


관찰

ch06_eda_summary.md가 notebooks\ch06\reports에 저장되었습니다.

가설

보고서 안의 표도 [27]과 같은 전체 상태 기준 값이라, 본문(completed 기준 148,990,000원)과 숫자가 다를 수 있습니다.

추가 검증

저장된 md의 카테고리 표에서 스포츠 매출이 50,174,000원(전체)인지 31,743,000원(completed)인지 확인합니다.

판단

보고서 "주의사항"에 집계 기준(completed, 기간, 부분 월)을 추가하고, completed 표로 다시 생성하기로 했습니다.

## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [31]:
import sys

# 프로젝트 루트를 모듈 검색 경로에 추가 (한 번만)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()

dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [32]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
3,스포츠,468,50174000,19.63
1,뷰티,376,47551000,18.60
5,전자기기,401,41003000,16.04
2,생활용품,390,34839000,13.63
4,식품,240,33597000,13.14


관찰

run_basic_eda()가 11개 결과표를 반환했습니다.
모듈의 category_sales는 스포츠 50,174,000원, 뷰티 47,551,000원 입니다.

가설

src/eda.py에는 completed 필터가 없어, 스크립트로 반복 실행해도 Gate 전 결과가 나올 수 있습니다.

추가 검증

src/eda.py에서 order_status를 사용하는 코드가 있는지 검색합니다.

판단

run_basic_eda()에 status='completed' 인자를 추가해, 노트북과 스크립트가 같은 기준으로 집계하도록 수정하기로 했습니다.

## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```
에러가 나서 python -m scripts.run_eda 로 실행
이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.

Q1. 전체 주문 중 취소·환불 비율은 얼마인가?

지표: 상태별 주문 비율 = 상태별 주문 수 / 전체 주문 수
데이터: orders
pandas: value_counts(normalize=True)

Q2. 월별 매출은 어떻게 변하는가?

지표: 월별 매출 = completed 주문의 line_total 합, 전월 대비 증감률(%)
데이터: orders + order_items
pandas: merge, 불리언 필터, groupby('order_month')['line_total'].sum(), pct_change()
주의: 2025-09, 2026-09는 부분 월이므로 표시하거나 제외합니다.

Q3. 어떤 카테고리가 매출과 판매수량을 이끄는가?

지표: 카테고리별 매출, 판매수량, 매출 비중(%), 카테고리 내 상품 1개당 매출
데이터: order_items + products + orders(상태 필터)
pandas: merge ×2, groupby('category').agg(revenue=('line_total','sum'), qty=('quantity','sum')), 비중 = revenue / revenue.sum()
참고: 카테고리별 상품 수가 7~19개로 차이가 큽니다. 그래서 총액과 함께 상품당 매출도 봐야 공정한 비교가 됩니다.

Q4. 결제수단에 따라 취소·환불률이 다른가?

지표: 결제수단별 completed / cancelled / refunded 비율
데이터: orders
pandas: pd.crosstab(o.payment_method, o.order_status, normalize='index')
참고: 결제수단별 주문은 70~79건으로 적습니다. 차이를 해석하려면 scipy.stats.chi2_contingency로 검정까지 해보는 것이 안전합니다.

Q5. 요일별 주문 건수와 주문당 금액은 어떻게 다른가?

지표: 요일별 주문 수, 요일별 평균 주문금액(AOV)
데이터: orders + order_items
pandas: 주문 단위 합계 groupby('order_id')['line_total'].sum() 후 merge, groupby('order_dayofweek').agg(...), 월~일 순서로 reindex

Q6. 한 번 주문할 때 얼마나, 몇 개를 사는가?

지표: 주문당 금액의 분포(평균·중앙값·사분위수), 주문당 품목 수(행 수), 주문당 총수량
데이터: order_items (+ orders 상태 필터)
pandas: groupby('order_id').agg(amount=('line_total','sum'), lines=('product_id','size'), units=('quantity','sum')), describe(), quantile(), hist()

Q7. 고객은 몇 번 주문하고, 재구매 고객은 얼마나 되는가?

지표:
구매 전환 고객 비율 = 주문이 있는 고객 / 전체 고객 (126/150)
고객당 주문 수 분포
재구매율 = completed 주문 2회 이상 고객 / completed 주문 1회 이상 고객
데이터: customers + orders
pandas: nunique(), groupby('customer_id').size(), value_counts(), merge(how='left') 후 fillna(0)

Q8. 성별·연령대·도시별로 1인당 매출에 차이가 있는가?

지표: 세그먼트별 고객 수, 주문 수, 1인당 매출(= 세그먼트 매출 / 세그먼트 고객 수)
데이터: customers + orders + order_items
pandas: pd.cut(age, bins=[18,29,39,49,59,69], labels=['20대 이하','30대','40대','50대','60대']), merge ×2, groupby(['gender','age_group']), pivot_table


Q9. 매출이 소수 고객에게 얼마나 집중되어 있는가?

지표: 상위 20% 고객의 매출 비중, 누적 매출 곡선(파레토)
데이터: orders + order_items
pandas: 고객별 매출 groupby('customer_id')['line_total'].sum(), sort_values(ascending=False), cumsum() / sum(), head(int(n*0.2))

Q10. 가입 후 첫 주문까지 얼마나 걸리는가? (데이터 정합성 점검 포함)

지표: 고객별 (첫 주문일 − 가입일)의 일수 분포, 그리고 가입일 이전 주문 건수와 비율
데이터: customers + orders
pandas: pd.to_datetime, groupby('customer_id')['order_date'].min(), merge, (first_order - signup_date).dt.days, (days < 0).sum()

```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


1. 관찰 

표에서 계산한 보조 지표(수량 비중, 개당 평균 매출)를 함께 정리하면 다음과 같습니다.

| 카테고리 | 판매수량 | 수량 비중 | 매출(원) | 매출 비중 | 개당 평균 매출(원)* |
|:---|---:|---:|---:|---:|---:|
| 전자기기 | 320 | 21.1% | 12,500,000 | 42.5% | 39,063 |
| 생활용품 | 510 | 33.6% | 7,800,000 | 26.5% | 15,294 |
| 패션 | 260 | 17.1% | 6,200,000 | 21.1% | 23,846 |
| 식품 | 430 | 28.3% | 2,900,000 | 9.9% | 6,744 |
| **합계** | **1,520** | **100.0%** | **29,400,000** | **100.0%** | **19,342** |

\* 개당 평균 매출 = total_sales ÷ total_quantity


매출 1위는 전자기기로 전체의 42.5%를 차지합니다. 수량 비중은 21.1%에 그쳐, 매출 비중이 수량 비중의 약 2배입니다.
판매수량 1위는 생활용품(510개, 33.6%)이지만 매출 비중은 26.5%로 2위입니다.
**식품은 수량 2위(28.3%)인데 매출은 최하위(9.9%)**입니다. 개당 평균 매출은 약 6,700원으로, 전자기기의 약 1/6 수준입니다.
패션은 수량(17.1%)과 매출(21.1%) 모두 중간권이며, 두 비중의 차이가 가장 작습니다.
상위 2개 카테고리(전자기기·생활용품)가 전체 매출의 69.0%를 차지합니다.

2. 가설 (표만으로는 검증되지 않은 설명)


| 검증 대상 | 분석 질문 | 필요한 지표 |
|:---|:---|:---|
| H1 단가 구조 | 카테고리별 실제 단가 분포는 어떤가? | 카테고리별 `unit_price` 평균·중앙값·범위 |
| H2 고가 상품 집중 | 카테고리 매출이 특정 상품에 얼마나 집중되어 있는가? | 카테고리 내 상위 3개 상품 매출 비중 |
| H3 반복 구매 | 카테고리별로 재구매 고객이 얼마나 되는가? | 카테고리별 재구매율, 고객당 구매 횟수 |
| H4 구색 차이 | 상품 수를 감안해도 순위가 유지되는가? | 상품 1개당 매출, 판매 실적 있는 상품 비율 |
| H5 집계 기준 | 취소·환불을 반영하면 결과가 달라지는가? | 카테고리별 취소·환불률, completed 기준 매출 |
| 추세 | 비중이 기간에 따라 안정적인가? | 월별 카테고리 매출 비중 |
| 고객 | 카테고리별 구매 고객층이 다른가? | 카테고리 × 성별·연령대별 매출 |

3. 추가 분석 질문

| 검증 대상 | 분석 질문 | 필요한 지표 |
|:---|:---|:---|
| H1 | 카테고리별 실제 단가 분포는 어떤가? | 카테고리별 `unit_price` 평균·중앙값·범위 |
| H2 | 카테고리 매출이 특정 상품에 얼마나 집중되어 있는가? | 카테고리 내 상위 상품 3개의 매출 비중 |
| H3 | 카테고리별로 재구매 고객이 얼마나 되는가? | 카테고리별 재구매율, 고객당 구매 횟수 |
| H4 | 상품 수를 감안해도 순위가 유지되는가? | 상품 1개당 매출, 판매 실적이 있는 상품 비율 |
| H5 | 취소·환불을 반영하면 결과가 달라지는가? | 카테고리별 취소·환불률, completed 기준 매출 |
| 추세 | 비중이 기간에 따라 안정적인가, 특정 월에 몰렸는가? | 월별 카테고리 매출 비중 |
| 고객 | 카테고리별 구매 고객층(성별·연령대)이 다른가? | 카테고리 × 세그먼트별 매출 |

4. 카테고리별 매출은 전자기기(42.5%), 생활용품(26.5%), 패션(21.1%), 식품(9.9%) 순이며, 상위 두 카테고리가 전체 매출의 69.0%를 차지한다. 판매수량 순위는 생활용품(33.6%), 식품(28.3%), 전자기기(21.1%), 패션(17.1%) 순으로 매출 순위와 일치하지 않는다. 특히 전자기기는 수량 비중 대비 매출 비중이 약 2배로 높은 반면, 식품은 수량 비중 대비 매출 비중이 약 1/3 수준이다. 이러한 차이는 카테고리 간 단가 수준 또는 상품 구성의 차이에서 비롯되었을 가능성이 있으나, 본 집계표만으로는 확인할 수 없다. 향후 카테고리별 단가 분포, 상품별 매출 집중도, 재구매율, 취소·환불률을 추가로 분석하여 이를 검증할 필요가 있다.

## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [38]:
import pandas as pd
from pathlib import Path

# CSV 위치를 직접 알면 여기에 적으세요. 예: Path(r"C:\dev\llm-data-analysis-study\data")
DATA_DIR = r"C:\dev\llm-data-analysis-study\data\processed"

if DATA_DIR is None:
    # 현재 폴더와 상위 폴더들에서 customers_clean.csv를 찾음 (.venv 폴더는 제외)
    found = None
    for base in [Path.cwd(), *Path.cwd().parents][:3]:
        hits = [p for p in base.rglob("customers_clean.csv") if ".venv" not in p.parts]
        if hits:
            found = hits[0].parent
            break
    if found is None:
        raise FileNotFoundError(
            f"customers_clean.csv를 찾지 못했습니다. 현재 폴더: {Path.cwd()}\n"
            "DATA_DIR에 CSV가 있는 폴더 경로를 직접 적어 주세요."
        )
    DATA_DIR = found

DATA_DIR = Path(DATA_DIR)
print("데이터 폴더:", DATA_DIR)

read = lambda f: pd.read_csv(DATA_DIR / f, encoding="utf-8-sig")
customers = read("customers_clean.csv")
products  = read("products_clean.csv")
orders    = read("orders_clean.csv")
items     = read("order_items_clean.csv")

데이터 폴더: C:\dev\llm-data-analysis-study\data\processed


In [40]:
order_amount = items.groupby("order_id", as_index=False)["line_total"].sum().rename(columns={"line_total": "order_amount"})
orders = orders.merge(order_amount, on="order_id", how="left")
done = orders[orders["order_status"] == "completed"]
items_done = items.merge(done[["order_id", "order_month"]], on="order_id").merge(products, on="product_id")
 
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [41]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
city_cust = customers.groupby("city").size().rename("customer_count")
city_sales = done.merge(customers[["customer_id", "city"]], on="customer_id").groupby("city")["order_amount"].sum().rename("total_sales")
t1 = pd.concat([city_cust, city_sales], axis=1).fillna(0)
t1["sales_per_customer"] = (t1["total_sales"] / t1["customer_count"]).round(0)
t1["customer_rank"] = t1["customer_count"].rank(ascending=False, method="min").astype(int)
t1["sales_rank"] = t1["total_sales"].rank(ascending=False, method="min").astype(int)
t1 = t1.sort_values("total_sales", ascending=False)
print("\n[1] 도시별 고객 수 vs 총 구매 금액\n", t1)
print("상관계수(고객 수 vs 총 구매 금액):", round(t1["customer_count"].corr(t1["total_sales"]), 3))
print("순위 상관(Spearman):", round(t1["customer_count"].corr(t1["total_sales"], method="spearman"), 3))


[1] 도시별 고객 수 vs 총 구매 금액
       customer_count  total_sales  sales_per_customer  customer_rank  sales_rank
city                                                                            
성남                21     24099000           1147571.0              1           1
고양                11     19116000           1737818.0             10           2
서울                15     19097000           1273133.0              4           3
울산                14     15380000           1098571.0              6           4
광주                17     14264000            839059.0              2           5
부산                16     13604000            850250.0              3           6
인천                14     12832000            916571.0              6           7
대전                14     11792000            842286.0              6           8
수원                13     10620000            816923.0              9           9
대구                15      8186000            545733.0              4          10
상관

In [42]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
cat_n = products.groupby("category").size().rename("product_count")
cat_sales = items_done.groupby("category")["line_total"].sum().rename("total_sales")
t2 = pd.concat([cat_n, cat_sales], axis=1).fillna(0)
t2["sales_per_product"] = (t2["total_sales"] / t2["product_count"]).round(0)
t2 = t2.sort_values("product_count", ascending=False)
print("\n[2] 카테고리별 상품 수 vs 매출\n", t2)
print("상관계수(상품 수 vs 매출):", round(t2["product_count"].corr(t2["total_sales"]), 3))
print("순위 상관(Spearman):", round(t2["product_count"].corr(t2["total_sales"], method="spearman"), 3))


[2] 카테고리별 상품 수 vs 매출
           product_count  total_sales  sales_per_product
category                                               
스포츠                  19     31743000          1670684.0
전자기기                 17     26400000          1552941.0
뷰티                   16     23383000          1461438.0
생활용품                 16     23915000          1494688.0
도서                   14     16389000          1170643.0
패션                   11     10587000           962455.0
식품                    7     16573000          2367571.0
상관계수(상품 수 vs 매출): 0.813
순위 상관(Spearman): 0.883


In [43]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.
t3 = done.groupby("payment_method")["order_amount"].agg(order_count="size", avg_order_value="mean", median_order_value="median").round(0)
t3 = t3.sort_values("avg_order_value", ascending=False)
print("\n[3] 결제수단별 평균 주문 금액 (completed)\n", t3)


[3] 결제수단별 평균 주문 금액 (completed)
                 order_count  avg_order_value  median_order_value
payment_method                                                  
naver_pay                51         852941.0            860000.0
card                     39         815538.0            837000.0
kakao_pay                49         802898.0            804000.0
bank_transfer            45         763156.0            713000.0


In [44]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.
t4 = items_done.pivot_table(index="order_month", columns="category", values="line_total", aggfunc="sum", fill_value=0, margins=True, margins_name="합계")
print("\n[4] 월별 x 카테고리별 매출 (단위: 천 원)\n", (t4 / 1000).round(0).astype(int))


[4] 월별 x 카테고리별 매출 (단위: 천 원)
 category        도서     뷰티   생활용품    스포츠     식품   전자기기     패션      합계
order_month                                                         
2025-09        410    580    893    269      0    498      0    2650
2025-10        774   3204   1162   2961   2481   4377   2011   16970
2025-11        844   1130   1610   3610    422   2909    496   11021
2025-12       1831   4282   4970   4630   1283   3158    931   21085
2026-01       1589   2061   2139   2800   1887   3232    564   14272
2026-02        582    481    259    489   1933    916    124    4784
2026-03       2957   1996   2318   5184   1895   3282    691   18323
2026-04       1536   2310   1193   2412   1457   1230   1408   11546
2026-05       1286   2330   2446   5408   2094   1161   1906   16631
2026-06       2310   2063   2660   1212   1889    815   1555   12504
2026-07       1928   2845   2126   2061    175   1389    264   10788
2026-08          0     69   1579    103    134   1843    637    4365
2026

In [45]:

#5 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
t5 = done.groupby("customer_id")["order_amount"].agg(order_count="size", avg_order_value="mean", total_sales="sum")
print("\n[5] 고객별 지표 분포\n", t5.describe().round(0))
print("order_count 분포:", t5["order_count"].value_counts().sort_index().to_dict())
aov_cut = t5["avg_order_value"].quantile(0.75)
print("AOV 75% 분위수:", round(aov_cut))
 
def segment(r):
    if r["order_count"] >= 2 and r["avg_order_value"] >= aov_cut:
        return "반복+고액"
    if r["order_count"] >= 2:
        return "반복 구매"
    if r["avg_order_value"] >= aov_cut:
        return "고액 단발"
    return "일반 단발"
 
t5["segment"] = t5.apply(segment, axis=1)
seg = t5.groupby("segment").agg(customers=("order_count", "size"), avg_orders=("order_count", "mean"), avg_aov=("avg_order_value", "mean"), sales=("total_sales", "sum"))
seg["sales_share_%"] = (seg["sales"] / seg["sales"].sum() * 100).round(1)
print("\n[5] 세그먼트 요약\n", seg.round(1))
print("completed 주문이 없는 고객 수:", customers.shape[0] - t5.shape[0])


[5] 고객별 지표 분포
        order_count  avg_order_value  total_sales
count        100.0            100.0        100.0
mean           2.0         825066.0    1489900.0
std            1.0         397371.0     958540.0
min            1.0          56000.0     112000.0
25%            1.0         528250.0     828750.0
50%            2.0         807000.0    1296500.0
75%            2.0        1061750.0    2008500.0
max            5.0        1931000.0    4100000.0
order_count 분포: {1: 44, 2: 37, 3: 12, 4: 5, 5: 2}
AOV 75% 분위수: 1061750

[5] 세그먼트 요약
          customers  avg_orders    avg_aov     sales  sales_share_%
segment                                                           
고액 단발           15         1.0  1354333.3  20315000           13.6
반복 구매           46         2.6   689447.5  84917000           57.0
반복+고액           10         2.0  1328100.0  26562000           17.8
일반 단발           29         1.0   592965.5  17196000           11.5
completed 주문이 없는 고객 수: 50


 6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.

 당신은 데이터 분석 질문을 검토하는 분석가입니다.
아래 데이터셋 스키마만 사용할 수 있습니다. 스키마에 없는 컬럼이나 외부 정보는 사용할 수 없습니다.

## 데이터셋 스키마
- customers: customer_id, name, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status(completed/cancelled/refunded), order_month, order_dayofweek
- order_items: order_item_id, order_id, product_id, quantity, unit_price, line_total
- 데이터 특성: 주문 기간 2025-09-15 ~ 2026-09-14, unit_price는 항상 price와 같음(할인 정보 없음)

## 작업
입력된 각 질문에 대해 다음을 수행하세요.
1. 판정: 현재 스키마로 "답할 수 있음 / 일부만 가능 / 답할 수 없음" 중 하나로 판정합니다.
2. 이유: 답할 수 없다면 어떤 정보(컬럼, 테이블, 기간)가 없는지 구체적으로 적습니다.
3. 대체 질문: 원래 질문의 의도에 가장 가까우면서 현재 스키마로 계산할 수 있는 질문 1~2개를 제안합니다.
4. 지표: 대체 질문을 계산식으로 표현합니다. 사용하는 테이블과 컬럼을 명시합니다.
5. 한계: 대체 질문이 원래 질문과 어떻게 다른지, 무엇을 알 수 없는지 한 문장으로 적습니다.

## 규칙
- 스키마에 없는 컬럼을 만들어 내지 마세요.
- 원인이나 동기는 데이터로 확인할 수 없습니다. 대체 질문은 "무엇이 관찰되는가" 형태로 작성하세요.
- 대리 지표(proxy)를 쓸 경우 반드시 "대리 지표"라고 표시하세요.

## 출력 형식
| 원래 질문 | 판정 | 이유 | 대체 질문 | 지표(계산식) | 한계 |

## 예시
| 원래 질문 | 판정 | 이유 | 대체 질문 | 지표(계산식) | 한계 |
|:---|:---|:---|:---|:---|:---|
| 할인 쿠폰이 매출을 늘렸는가? | 답할 수 없음 | 할인·쿠폰 컬럼 없음, unit_price = price | 카테고리별 단가 구간에 따라 판매수량이 다른가? | 단가 구간(pd.cut(unit_price))별 quantity 합계 | 가격 수준과 수량의 관계만 볼 수 있고, 할인 효과는 알 수 없음 |

## 검토할 질문
1. 고객들은 왜 주문을 취소하는가?
2. 어떤 상품의 이익률이 가장 높은가?
3. 사이트 방문자 중 몇 %가 구매하는가?

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.
